# Final Direct Preference Optimization (DPO) Experiment

This notebook runs the final DPO condition for the SFT-vs-DPO comparison. It uses the same shared data-preparation and evaluation utilities as the SFT notebook, guaranteeing that DPO trains on the same 5,000 underlying HH-RLHF training pairs and evaluates on the same 1,000 test pairs.


## 1. Install dependencies

Run this cell in Google Colab or a fresh environment. It intentionally does not install PyTorch, CUDA, or NumPy.


In [ ]:
%pip install -q \
    transformers==4.53.3 \
    datasets==3.6.0 \
    accelerate==1.8.1 \
    trl==0.19.1 \
    huggingface-hub==0.36.2 \
    fsspec==2025.3.0


## 2. Import shared project code


In [ ]:
from pathlib import Path
import os
import sys

# Keep framework selection explicit in Colab/local notebooks.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Shared src available:", (PROJECT_ROOT / "src").exists())


## 3. Environment and seed


In [ ]:
import torch
import transformers
import datasets
import accelerate
import trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Seed:", SEED)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Shared HH-RLHF split preparation

Parsing and validity filtering happen before deterministic sampling. This is the same function and configuration used by the SFT notebook.


In [ ]:
from transformers import AutoTokenizer
from src.data_preparation import (
    MODEL_NAME,
    DATASET_NAME,
    TRAIN_SIZE,
    TEST_SIZE,
    MAX_LENGTH,
    MAX_PROMPT_LENGTH,
    prepare_hh_rlhf_splits,
)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("Train pairs:", TRAIN_SIZE)
print("Test pairs:", TEST_SIZE)
print("Max sequence length:", MAX_LENGTH)
print("Max prompt length:", MAX_PROMPT_LENGTH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

default_tokenizer_length = getattr(tokenizer, "model_max_length", None)
print("Tokenizer pad token:", tokenizer.pad_token)
print("Tokenizer model_max_length:", default_tokenizer_length)

train_pairs, test_pairs, split_info = prepare_hh_rlhf_splits(tokenizer)

assert len(train_pairs) == TRAIN_SIZE
assert len(test_pairs) == TEST_SIZE
assert len(set(train_pairs["source_index"])) == TRAIN_SIZE
assert len(set(test_pairs["source_index"])) == TEST_SIZE

print("Split information:")
for key, value in split_info.as_dict().items():
    print(f"  {key}: {value}")

print("First five train source indices:", train_pairs["source_index"][:5])
print("First five test source indices: ", test_pairs["source_index"][:5])


## 5. Load the base policy model


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

param_count = sum(p.numel() for p in model.parameters())
print("Loaded model:", MODEL_NAME)
print("Parameter count:", f"{param_count:,}")
print("Dtype:", next(model.parameters()).dtype)


## 6. Build the DPO dataset from the shared training pairs


In [ ]:
from src.data_preparation import to_dpo_dataset

dpo_train = to_dpo_dataset(train_pairs)
assert len(dpo_train) == TRAIN_SIZE

print("DPO training pairs:", len(dpo_train))
print("Columns:", dpo_train.column_names)
print("Example prompt preview:", dpo_train[0]["prompt"][:200])
print("Example chosen preview:", dpo_train[0]["chosen"][:200])
print("Example rejected preview:", dpo_train[0]["rejected"][:200])


## 7. DPO training configuration


In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(
    output_dir="./smollm2_dpo",
    num_train_epochs=1,
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    beta=0.1,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    report_to="none",
    seed=SEED,
)

print("Epochs:", dpo_config.num_train_epochs)
print("Learning rate:", dpo_config.learning_rate)
print("Per-device train batch size:", dpo_config.per_device_train_batch_size)
print("Gradient accumulation:", dpo_config.gradient_accumulation_steps)
print("Effective batch size:", dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps)
print("Beta:", dpo_config.beta)
print("Max length:", dpo_config.max_length)
print("Max prompt length:", dpo_config.max_prompt_length)
print("FP16:", dpo_config.fp16)
print("BF16:", dpo_config.bf16)


## 8. Create the DPO trainer

`ref_model=None` makes TRL create the reference policy from the initial policy model, so the DPO reference is the frozen starting model rather than the separately trained SFT model.


In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    processing_class=tokenizer,
    train_dataset=dpo_train,
)

print("DPO trainer ready")
print("Reference model argument: None")
print("Training examples:", len(dpo_train))


## 9. Train DPO


In [ ]:
train_result = trainer.train()
print("DPO training complete")
print(train_result)


## 10. Save the DPO model


In [ ]:
SAVE_DIR = PROJECT_ROOT / "models" / "smollm2_dpo_final"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved DPO model:", SAVE_DIR)


## 11. Evaluate DPO on the shared test set

The evaluation function is identical to the Base and SFT evaluation: mean response-token log-probability, margin = chosen score minus rejected score, and margin > 0 is correct.


In [ ]:
from src.evaluation import evaluate_preference_pairs, print_preference_summary

dpo_rows, dpo_stats = evaluate_preference_pairs(
    model=model,
    tokenizer=tokenizer,
    pairs=test_pairs,
    output_csv=PROJECT_ROOT / "results" / "dpo_preference_eval.csv",
    model_label="DPO model",
    max_length=MAX_LENGTH,
)
print_preference_summary("DPO MODEL", dpo_stats)
print("Saved per-example CSV:", PROJECT_ROOT / "results" / "dpo_preference_eval.csv")
